In [2]:
import numpy as np
import pygame
import sys
import random

In [3]:
original_Layout = [
        "WWWWWWWWWWWWWWWWWWWW",
        "W.. ...W.....P.. ..W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W WWWW W.WWW.WWWWW WW",
        "W.WWWW.W.WWW.WWWWW.W",
        "W... .... O.. .....W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W. ....W.... .S....W",
        "WWWWWWWWWWWWWWWWWWWW"
    ]

    

In [4]:
def initialize_game():
    pygame.init()

    # Maze layout
    # W = Wall, . = Pellet, O = Obstacle, P = Purple Obstacle, S = Score-Lowering Obstacle, space = Empty
    maze_layout1 = [
        "WWWWWWWWWWWWWWWWWWWW",
        "W.. ...W.....P.. ..W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W WWWW W.WWW.WWWWW WW",
        "W.WWWW.W.WWW.WWWWW.W",
        "W... .... O.. .....W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W.WWWW.W.WWW.WWWWW.W",
        "W. ....W.... .S....W",
        "WWWWWWWWWWWWWWWWWWWW"
    ]

    maze_layout = [list(row) for row in maze_layout1]

    # Window dimensions and scale
    WIDTH, HEIGHT = 400, 200
    scale = 20
    # Window to set dimensions and scale
    win = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Pac-Man Maze")

    # Colors
    BLACK = (0, 0, 0)  # background
    BLUE = (0, 0, 255)  # wall color
    WHITE = (255, 255, 255)  # pellet color
    YELLOW = (255, 255, 0)  # pac man color
    RED = (255, 0, 0)  # red color for 5 Second penalty
    PURPLE = (128, 0, 128)  # Purple color for 10 Second penalty
    ORANGE = (255, 140, 0)  # Orange color for score-lowering obstacle

    return maze_layout, win, BLACK, BLUE, WHITE, YELLOW, RED, PURPLE, ORANGE, WIDTH, HEIGHT, scale


In [5]:
def reset_game(original_Layout):
    global pacman_x, pacman_y, score, timer_duration, start_time, game_over, maze_layout

    # Reset Pac-Man's position
    pacman_x, pacman_y = 1, 1

    # Reset game variables
    score = 0
    timer_duration = 20000  # Adjust as needed
    start_time = pygame.time.get_ticks()
    game_over = False

    # Reset the maze layout to the original one
    maze_layout = [row[:] for row in original_Layout]
    draw_maze(scale)
    pygame.display.update()


In [6]:
def get_state_representation():
    # Flatten the maze layout to represent the state
    flat_maze = [char for row in maze_layout for char in row]

    # Encode the state based on Pac-Man's position and the flattened maze
    state = pacman_y * len(maze_layout[0]) + pacman_x

    # Encode time as a feature (scaled to a reasonable range, e.g., 0 to 1)
    time_feature = (pygame.time.get_ticks() - start_time) / timer_duration
    state = state * 1000 + int(time_feature * 1000)

    # Encode obstacle presence as a feature (1 if there is an obstacle, 0 otherwise)
    obstacle_feature = int(any(char == 'O' for char in flat_maze))
    state = state * 2 + obstacle_feature

    # Ensure the state is within bounds
    state = max(0, min(state, state_space_size - 1))

    return state


In [7]:
def update_q_table(state, action, new_state, reward):
    # Update Q-value using Q-learning formula
    q_table[state][action] += alpha * (reward + gamma * np.max(q_table[new_state]) - q_table[state][action])



In [8]:
def choose_action(state, epsilon):
    # Choose action using epsilon-greedy policy
    if random.uniform(0, 1) < epsilon:
        action = random.randint(0, action_space_size - 1)
    else:
        action = np.argmax(q_table[state])
    return action



In [9]:
# GUI
def draw_maze(scale):
    win.fill(BLACK)
    for y, row in enumerate(maze_layout):
        for x, char in enumerate(row):
            screen_x, screen_y = x * scale, y * scale
            if char == 'W':
                pygame.draw.rect(win, BLUE, (screen_x, screen_y, scale, scale))
            elif char == '.':
                pygame.draw.circle(win, WHITE, (screen_x + scale // 2, screen_y + scale // 2), 3)
            elif char == 'O':
                pygame.draw.circle(win, RED, (screen_x + scale // 2, screen_y + scale // 2), 6)  # Red obstacle
            elif char == 'P':
                pygame.draw.circle(win, PURPLE, (screen_x + scale // 2, screen_y + scale // 2), 6)  # Purple obstacle
            elif char == 'S':
                pygame.draw.circle(win, ORANGE, (screen_x + scale // 2, screen_y + scale // 2), 6)  # Score-lowering orange obstacle
    # Drawing Pac-Man
    pygame.draw.circle(win, YELLOW, (pacman_x * scale + scale // 2, pacman_y * scale + scale // 2), scale // 2)
    # Drawing score and timer
    font = pygame.font.SysFont(None, 24)
    score_text = font.render(f"Score: {score}", True, WHITE)
    remaining_time = max(0, timer_duration - (pygame.time.get_ticks() - start_time))
    timer_color = RED if remaining_time <= 5000 else WHITE  # Change timer color to red when 5 seconds or less
    timer_text = font.render(f"Time: {remaining_time // 1000}s", True, timer_color)
    win.blit(score_text, (5, 0))
    win.blit(timer_text, (WIDTH - 100, 0))



In [10]:
def move_pacman(dx, dy):
    global pacman_x, pacman_y, score, timer_duration, maze_layout
    # Get the new position
    new_x, new_y = pacman_x + dx, pacman_y + dy

    # Check if the new position is within the maze boundaries
    if 0 <= new_x < len(maze_layout[0]) and 0 <= new_y < len(maze_layout):
        char_at_new_pos = maze_layout[new_y][new_x]

        # Check for walls
        if char_at_new_pos != 'W':
            if char_at_new_pos == '.':
                # Update maze layout to remove pellet
                maze_row_list = list(maze_layout[new_y])
                maze_row_list[new_x] = ' '
                maze_layout[new_y] = ''.join(maze_row_list)
                score += 10 
                
            elif char_at_new_pos == 'O' or char_at_new_pos == 'P' or char_at_new_pos == 'S':
                # Update maze layout to remove obstacle
                maze_row_list = list(maze_layout[new_y])
                maze_row_list[new_x] = ' '
                maze_layout[new_y] = ''.join(maze_row_list)
                
                if char_at_new_pos == 'O':
                    timer_duration -= 5000
                elif char_at_new_pos == 'P':
                    timer_duration -= 10000
                elif char_at_new_pos == 'S':
                    score = max(0, score - 50) 

            pacman_x, pacman_y = new_x, new_y


In [11]:
def main_game_loop():
    global pacman_x, pacman_y, score, timer_duration, start_time, game_over
    intial_epsilon= 1.0
    min_epsilon=0.01
    epsilon_decay=0.65


    running = True
    while running:
        current_time = pygame.time.get_ticks()
        remaining_time = max(0, timer_duration - (current_time - start_time))

        if remaining_time <= 0:
            game_over = True

        state = get_state_representation()
        epsilon=max(min_epsilon,intial_epsilon*epsilon_decay)
        # Choose action using epsilon-greedy policy
        action = choose_action(state, epsilon)

        dx, dy = 0, 0
        if action == 0:
            dy = -1  # Up
        elif action == 1:
            dy = 1  # Down
        elif action == 2:
            dx = -1  # Left
        elif action == 3:
            dx = 1  # Right

        new_x, new_y = pacman_x + dx, pacman_y + dy
        # Boundary Check
        if 0 <= new_x < len(maze_layout[0]) and 0 <= new_y < len(maze_layout):
            char_at_new_pos = maze_layout[new_y][new_x]
            if char_at_new_pos != 'W':
                move_pacman(dx, dy)

        # Get the next state
        new_state = get_state_representation()

        # Calculate the immediate reward (modify based on your reward structure)
        reward = 0
        if char_at_new_pos == '.':
            reward = 100
        elif char_at_new_pos == 'O':
            reward = -500
        elif char_at_new_pos == 'P':
            reward = -100
        elif char_at_new_pos == 'S':
            reward = -500

        # Update Q-value using Q-learning formula
        update_q_table(state, action, new_state, reward)

        draw_maze(20)
        pygame.display.update()
        # Check for game over
        if game_over:
            # Reset game state
            print(f"This run score was: {score}")
            pacman_x, pacman_y = 1, 1
            score = 0
            reset_game(original_Layout)
            start_time = pygame.time.get_ticks()
            game_over = False

        pygame.time.Clock().tick(30)  # Limit the frames per second



In [12]:
if __name__ == "__main__":
    maze_layout, win, BLACK, BLUE, WHITE, YELLOW, RED, PURPLE, ORANGE, WIDTH, HEIGHT, scale = initialize_game()

    # Q-learning parameters
    alpha = 0.1  # Learning rate
    gamma = 0.05  # Discount factor
    epsilon = 0.9  # Exploration-exploitation trade-off parameter

    # Q-table initialization
    state_space_size = len(maze_layout) * len(maze_layout[0])  
    action_space_size = 4  # 4 Directions
    q_table = np.zeros((state_space_size, action_space_size))

    # Initialize game state
    pacman_x, pacman_y = 1, 1
    score = 0
    timer_duration = 20000  # 20 seconds
    start_time = pygame.time.get_ticks()
    game_over = False

    main_game_loop()

This run score was: 320
This run score was: 250
This run score was: 390
This run score was: 200
This run score was: 200
This run score was: 380
This run score was: 340
This run score was: 260
This run score was: 230
This run score was: 360
This run score was: 280
This run score was: 520
This run score was: 340


KeyboardInterrupt: 